In [1]:
import json
import re
import numpy as np
import torch
import torch.nn as nn
import joblib
import pandapower as pp
import pandapower.networks as pn
import pandas as pd
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# Robustly find and move to the project root, regardless of where the kernel started
target_folder = "grid-congestion-project"
current = os.getcwd()

if os.path.basename(current) != target_folder:
    # Try the known direct path first
    known_path = r"C:\Users\HP\Music\grid-congestion-project"
    if os.path.isdir(known_path):
        os.chdir(known_path)
    else:
        # Fallback: walk upward looking for a folder containing 'app' and 'experiments'
        probe = current
        for _ in range(5):
            if os.path.isdir(os.path.join(probe, "app")) and os.path.isdir(os.path.join(probe, "experiments")):
                os.chdir(probe)
                break
            probe = os.path.dirname(probe)

print("Working dir:", os.getcwd())
print("Imports OK")

Working dir: C:\Users\HP\Music\grid-congestion-project
Imports OK


In [ ]:
import sys, os, json, random
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import pandapower as pp
import ipywidgets as widgets
from IPython.display import display, clear_output

sys.path.insert(0, os.path.abspath("src/simulation"))
sys.path.insert(0, os.path.abspath("src/training"))

from generate_scenarios_multitopology import TOPOLOGY_CONFIGS

# ============================================================
# Which model performed better per topology, based on validation F1
# from your training runs. Used to flag a "recommended" prediction.
# ============================================================
PREFERRED_MODEL = {
    "us_ieee118": "model2",
    "uk_ieee30": "model1",
    "fr_pegase89": "model1",
    "gb_full": "model1",
    "eu_case300": "model1",
    "india_synthetic": "model1",
}

# ============================================================
# Model 2 architecture (must match training exactly)
# ============================================================
class MultiModalCongestionModel(nn.Module):
    def __init__(self, vocab_size, struct_dim=9, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.struct_branch = nn.Sequential(
            nn.Linear(struct_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.text_lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
        )

    def forward(self, structured, text_seq):
        struct_out = self.struct_branch(structured)
        embedded = self.embedding(text_seq)
        _, (hidden, _) = self.text_lstm(embedded)
        text_out = hidden.squeeze(0)
        combined = torch.cat([struct_out, text_out], dim=1)
        return self.fusion(combined).squeeze(1)


def find_existing(*candidates):
    """Return the first path in candidates that actually exists on disk."""
    for c in candidates:
        if os.path.isfile(c):
            return c
    return None


def load_models_for_topology(topology):
    """Loads Model 1, Model 2 + vocab for a topology, trying both the
    suffixed filename pattern (used by all six multitopology runs) and
    the original unsuffixed pattern (us_ieee118's original files)."""
    m1_path = find_existing(f"app/model1_baseline_{topology}.pkl", "app/model1_baseline.pkl")
    m2_path = find_existing(f"app/model2_multimodal_{topology}.pt", "app/model2_multimodal.pt")
    vocab_path = find_existing(f"app/vocab_{topology}.json", "app/vocab.json")

    if not (m1_path and m2_path and vocab_path):
        raise FileNotFoundError(
            f"Missing files for '{topology}'. m1={m1_path}, m2={m2_path}, vocab={vocab_path}"
        )

    model1 = joblib.load(m1_path)

    with open(vocab_path, "r") as f:
        vocab = json.load(f)

    model2 = MultiModalCongestionModel(vocab_size=len(vocab))
    model2.load_state_dict(torch.load(m2_path, map_location="cpu"))
    model2.eval()

    return model1, model2, vocab, m1_path, m2_path


def tokenize(text):
    import re
    return re.findall(r"[a-z]+", text.lower())


def text_to_sequence(text, vocab, max_len=30):
    tokens = tokenize(text)
    seq = [vocab.get(tok, vocab.get("<UNK>", 1)) for tok in tokens]
    seq = seq[:max_len]
    seq = seq + [vocab.get("<PAD>", 0)] * (max_len - len(seq))
    return seq


def build_scenario(wind_speed, solar_irradiance, load_multiplier, has_outage, cfg):
    storm_flag = wind_speed > 65
    n_buses = cfg["n_renewable_buses"]
    wind_mid = sum(cfg["wind_mw_range"]) / 2
    solar_mid = sum(cfg["solar_mw_range"]) / 2
    total_wind_mw = n_buses * (wind_speed / 90.0) * wind_mid
    total_solar_mw = n_buses * (solar_irradiance / 1000.0) * solar_mid

    event = {"type": "none"}
    if has_outage:
        cause = "storm" if storm_flag else "equipment_failure"
        event = {"type": "line_outage", "line_id": None, "cause": cause}

    text_parts = [f"Wind speed measured at {wind_speed:.0f} km/h"]
    if event["type"] == "line_outage":
        text_parts.append(f"a line went offline due to {event['cause'].replace('_', ' ')}")
    text_parts.append(f"Load level at {load_multiplier:.2f}x baseline")
    text_report = "Grid alert: " + ". ".join(text_parts) + "."

    return {
        "weather": {"wind_speed_kmh": wind_speed, "solar_irradiance_wm2": solar_irradiance, "storm_flag": storm_flag},
        "load_multiplier": load_multiplier,
        "event": event,
        "total_wind_mw": total_wind_mw,
        "total_solar_mw": total_solar_mw,
        "text_report": text_report,
    }


def extract_structured_features_raw(scenario):
    """9 features, same order as training -- unnormalized, for Model 1 (RF)."""
    w = scenario["weather"]
    e = scenario["event"]
    return [
        w["wind_speed_kmh"], w["solar_irradiance_wm2"], int(w["storm_flag"]),
        scenario["load_multiplier"], scenario["total_wind_mw"], scenario["total_solar_mw"],
        1 if e["type"] == "line_outage" else 0,
        1 if e.get("cause") == "storm" else 0,
        1 if e.get("cause") == "equipment_failure" else 0,
    ]


def extract_structured_features_norm(scenario):
    """Same 9 features, normalized -- for Model 2 (LSTM), matching training."""
    w = scenario["weather"]
    e = scenario["event"]
    return [
        w["wind_speed_kmh"] / 90.0, w["solar_irradiance_wm2"] / 1000.0, int(w["storm_flag"]),
        (scenario["load_multiplier"] - 0.8) / 1.4,
        scenario["total_wind_mw"] / 240.0, scenario["total_solar_mw"] / 160.0,
        1 if e["type"] == "line_outage" else 0,
        1 if e.get("cause") == "storm" else 0,
        1 if e.get("cause") == "equipment_failure" else 0,
    ]


def rebuild_network_for_redispatch(scenario, cfg):
    net = cfg["loader"]()
    net.line['max_i_ka'] = net.line['max_i_ka'] * cfg["thermal_derate"]

    gen_buses = net.gen['bus'].tolist()
    n_buses = min(cfg["n_renewable_buses"], len(gen_buses))
    renewable_buses = gen_buses[:n_buses]
    per_bus_mw = (scenario["total_wind_mw"] + scenario["total_solar_mw"]) / max(n_buses, 1)

    for bus in renewable_buses:
        max_cap = max(per_bus_mw * 4.0, 50)
        pp.create_sgen(net, bus=bus, p_mw=per_bus_mw, name=f"renewable_bus{bus}",
                        controllable=True, min_p_mw=0, max_p_mw=max_cap,
                        min_q_mvar=-max_cap * 0.3, max_q_mvar=max_cap * 0.3)

    net.load['p_mw'] = net.load['p_mw'] * scenario["load_multiplier"]
    net.load['q_mvar'] = net.load['q_mvar'] * scenario["load_multiplier"]

    if scenario["event"]["type"] == "line_outage":
        in_service_lines = net.line[net.line['in_service']].index.tolist()
        if in_service_lines:
            outage_line = random.choice(in_service_lines)
            net.line.at[outage_line, 'in_service'] = False

    return net


def setup_generator_costs(net):
    # Real grid data (e.g. case89pegase) often lacks bus voltage limits,
    # which can make OPF fail outright. Set safe, generous bounds.
    net.bus['min_vm_pu'] = 0.9
    net.bus['max_vm_pu'] = 1.1

    net.poly_cost = net.poly_cost.iloc[0:0]
    for idx in net.gen.index:
        net.gen.at[idx, 'controllable'] = True
        net.gen.at[idx, 'min_p_mw'] = 0
        net.gen.at[idx, 'max_p_mw'] = max(net.gen.at[idx, 'max_p_mw'], net.gen.at[idx, 'p_mw'] * 1.2 + 1)
        if pd.isna(net.gen.at[idx, 'min_q_mvar']):
            net.gen.at[idx, 'min_q_mvar'] = -net.gen.at[idx, 'max_p_mw'] * 0.5
        if pd.isna(net.gen.at[idx, 'max_q_mvar']):
            net.gen.at[idx, 'max_q_mvar'] = net.gen.at[idx, 'max_p_mw'] * 0.5
        pp.create_poly_cost(net, idx, 'gen', cp1_eur_per_mw=100)
    for idx in net.sgen.index:
        pp.create_poly_cost(net, idx, 'sgen', cp1_eur_per_mw=5)


# ============================================================
# UI
# ============================================================
topology_dropdown = widgets.Dropdown(
    options=list(TOPOLOGY_CONFIGS.keys()), value="us_ieee118", description="Topology:"
)
wind_slider = widgets.FloatSlider(value=15.0, min=0, max=90, step=1, description="Wind (km/h):")
solar_slider = widgets.FloatSlider(value=500.0, min=0, max=1000, step=10, description="Solar (W/m²):")
load_slider = widgets.BoundedFloatText(value=1.0, min=0.5, max=2.2, step=0.05, description="Load mult:")
outage_checkbox = widgets.Checkbox(value=False, description="Line outage event")
run_button = widgets.Button(description="Run Prediction & Redispatch", button_style="info", icon="search")
output_area = widgets.Output()

_model_cache = {}

def on_run_clicked(b):
    with output_area:
        clear_output()
        topology = topology_dropdown.value
        cfg = TOPOLOGY_CONFIGS[topology]

        print(f"DEBUG: wind_slider.value={wind_slider.value}, solar_slider.value={solar_slider.value}, load_slider.value={load_slider.value}, outage_checkbox.value={outage_checkbox.value}")

        if topology not in _model_cache:
            print(f"Loading models for {topology}...")
            _model_cache[topology] = load_models_for_topology(topology)
        model1, model2, vocab, m1_path, m2_path = _model_cache[topology]

        scenario = build_scenario(wind_slider.value, solar_slider.value, load_slider.value,
                                   outage_checkbox.value, cfg)

        print(f"Topology: {topology}")
        print(f"Scenario text: {scenario['text_report']}\n")

        # --- Model 1 ---
        X1 = np.array([extract_structured_features_raw(scenario)])
        m1_pred = model1.predict(X1)[0]
        m1_conf = model1.predict_proba(X1)[0][int(m1_pred)]

        # --- Model 2 ---
        struct_t = torch.tensor([extract_structured_features_norm(scenario)], dtype=torch.float32)
        text_seq_t = torch.tensor([text_to_sequence(scenario["text_report"], vocab)], dtype=torch.long)
        with torch.no_grad():
            logit = model2(struct_t, text_seq_t)
            m2_conf_raw = torch.sigmoid(logit).item()
        m2_pred = 1 if m2_conf_raw > 0.5 else 0
        m2_conf = m2_conf_raw if m2_pred == 1 else 1 - m2_conf_raw

        recommended = PREFERRED_MODEL.get(topology, "model1")

        print("--- Predictions ---")
        print(f"Model 1 (Structured Only):   {'CONGESTED' if m1_pred else 'NOT CONGESTED'} ({m1_conf*100:.1f}% confidence)")
        print(f"Model 2 (Structured + Text): {'CONGESTED' if m2_pred else 'NOT CONGESTED'} ({m2_conf*100:.1f}% confidence)")
        print(f"Recommended for this topology: {recommended} (higher validation F1 on {topology})\n")

        # --- Redispatch (only meaningful if congestion predicted) ---
        print("--- Redispatch Optimization ---")
        try:
            net = rebuild_network_for_redispatch(scenario, cfg)
            pp.runpp(net, enforce_q_lims=False, max_iteration=30, numba=False, init="flat")
            loading_before = net.res_line['loading_percent'].max()
            conv_before = net.gen['p_mw'].sum()
            ren_before = net.sgen['p_mw'].sum()
            total_before = conv_before + ren_before
            ren_pct_before = (ren_before / total_before * 100) if total_before > 0 else 0

            setup_generator_costs(net)
            pp.runopp(net, numba=False)
            loading_after = net.res_line['loading_percent'].max()
            resolved = loading_after <= 100

            conv_after = net.res_gen['p_mw'].sum()
            ren_after = net.res_sgen['p_mw'].sum()
            total_after = conv_after + ren_after
            ren_pct_after = (ren_after / total_after * 100) if total_after > 0 else 0

            print(f"Line Loading:     {loading_before:.2f}% -> {loading_after:.2f}% ({loading_after-loading_before:+.1f}%)")
            print(f"Renewable Usage:  {ren_pct_before:.1f}% -> {ren_pct_after:.1f}% ({ren_pct_after-ren_pct_before:+.1f}%)")
            print(f"Congestion Status: {'RESOLVED' if resolved else 'STILL CONGESTED'}")
        except Exception as e:
            import traceback
            print(f"Redispatch could not complete: {str(e)[:200]}")
            print("\nFull traceback for debugging:")
            print(traceback.format_exc())

run_button.on_click(on_run_clicked)

def on_load_change(change):
    with output_area:
        print(f"LIVE: load_slider changed to {change['new']}")
load_slider.observe(on_load_change, names='value')

display(widgets.VBox([
    topology_dropdown, wind_slider, solar_slider, load_slider, outage_checkbox,
    run_button, output_area
]))